# EDA - Proyecto Inmuebles

Notebook para analisis exploratorio sobre el dataset final para BI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

In [ ]:
DATA_PATH = '../pre-procesamiento/processed_powerbi_es.csv'
df = pd.read_csv(DATA_PATH, sep=';', decimal=',', low_memory=False)

print('Shape:', df.shape)
print('Columnas:', len(df.columns))
display(df.head())

Shape: (144961, 39)
Columnas: 39


,country,location,latitude,longitude,un_continent,un_statistical,price_in_Euro,entry_price_eur,building_construction_year,apartament_m2,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,riesgo_ocupacion,leyes_de_ocupacion_comentario,leyes_de_ocupacion_url,pib_millones_ppa,gini,gini_year,population_2018,population_2019,population_change,crime_index,safety_index,unemployment_last,unemployment_previous,unemployment_reference,country_code,country_iso_code,price_in_Euro_calculo,apartment_total_area_calculo,apartment_living_area_calculo,image,url,country_norm_merge_key
0,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,279.000 €,83.700 €,2022,mansion,3,5,3,2,1,280 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,279000.00,280.0,86.0,https://realting.com/uploads/bigSlider/493/0af...,https://realting.com/property-for-sale/armenia...,armenia
1,Armenia,"Dilijan, Tavush Province, Ijevan Region",40.069099,45.038189,Asia,Western Asia,279.000 €,83.700 €,2022,mansion,1,1,3,2,1,6 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,279000.00,6.0,86.0,https://realting.com/uploads/bigSlider/e12/552...,https://realting.com/property-for-sale/armenia...,armenia
2,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,"160.158,09 €","48.047,43 €",2022,apartment,2,1,2,1,1,80 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,160158.09,80.0,86.0,https://realting.com/uploads/bigSlider/048/1d1...,https://realting.com/property-for-sale/armenia...,armenia
3,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,167.400 €,50.220 €,2022,apartment,14,14,3,2,1,96 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,167400.00,96.0,86.0,https://realting.com/uploads/bigSlider/712/1aa...,https://realting.com/property-for-sale/armenia...,armenia
4,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,344.100 €,103.230 €,2022,apartment,9,8,5,4,1,124 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,344100.00,124.0,86.0,https://realting.com/uploads/bigSlider/fb0/723...,https://realting.com/property-for-sale/armenia...,armenia


In [4]:
# KPI 1: Total de inmuebles
total_inmuebles = int(len(df))
print(f'Total de inmuebles: {total_inmuebles:,}')

Total de inmuebles: 144,961


In [5]:
# KPI 2: Total de inmuebles por nivel de riesgo
riesgo_col = 'riesgo_ocupacion'

if riesgo_col in df.columns:
    riesgo_counts = (
        df[riesgo_col]
        .fillna('sin_dato')
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'': 'sin_dato'})
        .value_counts()
    )

    total_alto = int(riesgo_counts.get('alto', 0))
    total_medio = int(riesgo_counts.get('medio', 0))
    total_bajo = int(riesgo_counts.get('bajo', 0))

    print(f'Total riesgo alto: {total_alto:,}')
    print(f'Total riesgo medio: {total_medio:,}')
    print(f'Total riesgo bajo: {total_bajo:,}')

    display(riesgo_counts.rename_axis('riesgo_ocupacion').reset_index(name='total'))
else:
    print("No existe la columna 'riesgo_ocupacion' en el dataset.")

Total riesgo alto: 36,093
Total riesgo medio: 51,725
Total riesgo bajo: 57,143


,riesgo_ocupacion,total
0,bajo,57143
1,medio,51725
2,alto,36093


In [6]:
# KPI 3: Lista de paises con porcentaje de criminalidad (crime_index)
if {'country', 'crime_index'}.issubset(df.columns):
    crime_country = (
        df.assign(crime_index_num=pd.to_numeric(df['crime_index'], errors='coerce'))
          .dropna(subset=['country', 'crime_index_num'])
          .groupby('country', as_index=False)
          .agg(
              porcentaje_criminalidad=('crime_index_num', 'mean'),
              total_inmuebles=('country', 'size')
          )
          .sort_values('porcentaje_criminalidad', ascending=False)
          .reset_index(drop=True)
    )

    crime_country['porcentaje_criminalidad'] = crime_country['porcentaje_criminalidad'].round(2)

    print(f'Paises evaluados: {len(crime_country)}')
    display(crime_country)
else:
    print("No existen columnas requeridas: 'country' y/o 'crime_index'.")

Paises evaluados: 27


,country,porcentaje_criminalidad,total_inmuebles
0,United States,47.70,414
1,Indonesia,46.06,102
2,Italy,44.24,3002
3,Australia,41.67,1
4,Montenegro,41.56,10308
5,Greece,41.30,12341
6,Russia,40.60,18481
7,Thailand,40.01,1968
8,Turkey,39.50,25648
9,Northern Cyprus,39.50,745


In [ ]:
# KPI 4: Paises mas baratos con menor riesgo de ocupacion
required_cols = {'country', 'riesgo_ocupacion', 'price_in_Euro_calculo'}

if required_cols.issubset(df.columns):
    work = df.copy()
    work['precio_eur_num'] = pd.to_numeric(work['price_in_Euro_calculo'], errors='coerce')
    work['riesgo_norm'] = (
        work['riesgo_ocupacion']
        .fillna('sin_dato')
        .astype(str)
        .str.strip()
        .str.lower()
    )

    agg = (
        work.dropna(subset=['country', 'precio_eur_num'])
            .groupby('country', as_index=False)
            .agg(
                total_inmuebles=('country', 'size'),
                precio_medio_eur=('precio_eur_num', 'mean'),
                pct_riesgo_bajo=('riesgo_norm', lambda s: (s.eq('bajo').mean() * 100)),
                pct_riesgo_medio=('riesgo_norm', lambda s: (s.eq('medio').mean() * 100)),
                pct_riesgo_alto=('riesgo_norm', lambda s: (s.eq('alto').mean() * 100)),
            )
    )

    # Candidatos con riesgo bajo dominante
    candidatos = agg[agg['pct_riesgo_bajo'] >= 50].copy()

    ranking_barato_seguro = (
        candidatos.sort_values(['precio_medio_eur', 'pct_riesgo_alto', 'pct_riesgo_bajo'], ascending=[True, True, False])
                  .reset_index(drop=True)
    )

    for c in ['precio_medio_eur', 'pct_riesgo_bajo', 'pct_riesgo_medio', 'pct_riesgo_alto']:
        ranking_barato_seguro[c] = ranking_barato_seguro[c].round(2)

    top10 = ranking_barato_seguro.head(10)

    print('Top 10 paises mas baratos con menor riesgo de ocupacion (riesgo bajo >= 50%):')
    print(top10[['country', 'precio_medio_eur', 'pct_riesgo_bajo', 'pct_riesgo_alto', 'total_inmuebles']].to_string(index=False))

    # Segunda tabla simplificada (sin country, total_inmuebles ni precio_medio_eur)
    display(top10[['pct_riesgo_bajo', 'pct_riesgo_medio', 'pct_riesgo_alto']])
else:
    print('Faltan columnas requeridas para este analisis.')

Top 10 paises mas baratos con menor riesgo de ocupacion (riesgo bajo >= 50%):
        country  precio_medio_eur  pct_riesgo_bajo  pct_riesgo_alto  total_inmuebles
        Belarus          66827.11            100.0              0.0            13742
        Georgia          87222.28            100.0              0.0             3291
     Uzbekistan         104856.49            100.0              0.0             1950
      Lithuania         163625.71            100.0              0.0             2567
Northern Cyprus         174580.62            100.0              0.0              745
         Cyprus         181898.17            100.0              0.0               79
        Finland         207512.72            100.0              0.0              816
 Czech Republic         255896.28            100.0              0.0             1371
       Thailand         261075.72            100.0              0.0             1968
      Indonesia         319930.24            100.0              0.0     

,country,total_inmuebles,precio_medio_eur,pct_riesgo_bajo,pct_riesgo_medio,pct_riesgo_alto
0,Belarus,13742,66827.11,100.0,0.0,0.0
1,Georgia,3291,87222.28,100.0,0.0,0.0
2,Uzbekistan,1950,104856.49,100.0,0.0,0.0
3,Lithuania,2567,163625.71,100.0,0.0,0.0
4,Northern Cyprus,745,174580.62,100.0,0.0,0.0
5,Cyprus,79,181898.17,100.0,0.0,0.0
6,Finland,816,207512.72,100.0,0.0,0.0
7,Czech Republic,1371,255896.28,100.0,0.0,0.0
8,Thailand,1968,261075.72,100.0,0.0,0.0
9,Indonesia,102,319930.24,100.0,0.0,0.0


## Top 20 paises en procesamiento (modo rapido, sin verificacion web)
Filtro objetivo:
- Precio total entre 30.000 y 45.000 EUR (no entrada)
- Renta mensual >= 1.200 EUR
- Exclusion local de sold/outdated por texto

In [ ]:
import re


def parse_currency_local(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace('€', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.replace(' ', '', regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors='coerce')

In [ ]:
# Construir base de candidatos con filtros objetivo
top_work = df.copy()

if 'price_eur_num' not in top_work.columns:
    if 'price_in_Euro_calculo' in top_work.columns:
        top_work['price_eur_num'] = pd.to_numeric(top_work['price_in_Euro_calculo'], errors='coerce')
    else:
        top_work['price_eur_num'] = parse_currency_local(top_work['price_in_Euro'])

if 'rent_eur_month_num' not in top_work.columns:
    top_work['rent_eur_month_num'] = parse_currency_local(top_work['zone_rental_price'])

top_work['price_eur_num'] = pd.to_numeric(top_work['price_eur_num'], errors='coerce')
top_work['rent_eur_month_num'] = pd.to_numeric(top_work['rent_eur_month_num'], errors='coerce')

listing_text = (
    top_work.get('location', pd.Series(index=top_work.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + top_work.get('url', pd.Series(index=top_work.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + top_work.get('image', pd.Series(index=top_work.index, dtype='object')).fillna('').astype(str).str.lower()
)
sold_pattern = r'\bsold\b|\bvendido\b|\bvendida\b|\breservado\b|\breserved\b'
inactive_pattern = r'not\s*found|404|unavailable|off[- ]?market|no\s*found'

top_work = top_work[
    top_work.get('url', pd.Series(index=top_work.index, dtype='object')).fillna('').astype(str).str.contains(r'^https?://', regex=True)
    & (~listing_text.str.contains(sold_pattern, regex=True, na=False))
    & (~listing_text.str.contains(inactive_pattern, regex=True, na=False))
    & top_work['country'].notna()
    & top_work['price_eur_num'].between(30000, 45000, inclusive='both')
    & (top_work['rent_eur_month_num'] >= 1200)
].copy()

print(f'Candidatos tras filtro base (30k-45k y renta>=1200): {len(top_work):,}')
print(f'Paises candidatos: {top_work["country"].nunique():,}')

In [ ]:
# TOP 20 RAPIDO RECONFIGURADO 
import pandas as pd
import numpy as np

data_path = '../pre-procesamiento/processed.csv'
quick_df = pd.read_csv(data_path, sep=',', low_memory=False)

def parse_currency_quick(series: pd.Series) -> pd.Series:
    s = series.fillna('').astype(str).str.strip()
    # Intenta parseo directo primero
    direct = pd.to_numeric(s, errors='coerce')

    # Fallback para formatos con euro, miles y coma decimal
    cleaned = (
        s.str.replace('€', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace('.', '', regex=False)
         .str.replace(',', '.', regex=False)
    )
    fallback = pd.to_numeric(cleaned, errors='coerce')

    return direct.fillna(fallback)

# Precio TOTAL (no entrada)
if 'price_in_Euro_calculo' in quick_df.columns:
    quick_df['price_eur_num'] = pd.to_numeric(quick_df['price_in_Euro_calculo'], errors='coerce')
else:
    quick_df['price_eur_num'] = parse_currency_quick(quick_df.get('price_in_Euro', pd.Series(index=quick_df.index, dtype='object')))

quick_df['rent_eur_month_num'] = parse_currency_quick(
    quick_df.get('zone_rental_price', pd.Series(index=quick_df.index, dtype='object'))
)

listing_text = (
    quick_df.get('location', pd.Series(index=quick_df.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + quick_df.get('url', pd.Series(index=quick_df.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + quick_df.get('image', pd.Series(index=quick_df.index, dtype='object')).fillna('').astype(str).str.lower()
)
sold_pattern = r'\bsold\b|\bvendido\b|\bvendida\b|\breservado\b|\breserved\b'

# Filtro rapido pedido
q = quick_df[
    (~listing_text.str.contains(sold_pattern, regex=True, na=False))
    & quick_df['country'].notna()
    & quick_df['price_eur_num'].between(30000, 45000, inclusive='both')
    & (quick_df['rent_eur_month_num'] >= 1200)
].copy()

if q.empty:
    print('No hay resultados con: no sold + precio 30k-45k + renta >=1200')
else:
    # Score simple: cercano a 30k + renta alta
    q['score_precio_30k'] = (1.0 - (q['price_eur_num'] - 30000).abs() / 30000).clip(0.0, 1.0)
    q['score_renta_1200'] = ((q['rent_eur_month_num'] - 1200) / 1200).clip(lower=0.0, upper=2.0) / 2.0
    q['score_objetivo'] = 0.7 * q['score_precio_30k'] + 0.3 * q['score_renta_1200']

    top20_rapido = (
        q.sort_values(['score_objetivo', 'rent_eur_month_num'], ascending=[False, False])
         .groupby('country', as_index=False).head(1)
         .sort_values(['score_objetivo', 'rent_eur_month_num'], ascending=[False, False])
         .head(20)
         .reset_index(drop=True)
    )
    top20_rapido.insert(0, 'rank_top20', np.arange(1, len(top20_rapido) + 1))

    cols = ['rank_top20', 'country', 'location', 'price_eur_num', 'rent_eur_month_num', 'score_objetivo', 'url']
    display(top20_rapido[cols])
    print(f'Resultados: {len(top20_rapido)} | Paises unicos: {top20_rapido["country"].nunique()}')


,rank_top20,country,location,price_eur_num,rent_eur_month_num,score_objetivo,url
0,1,Greece,"Macedonia and Thrace, Municipality of Paggaio,...",30203.61,2527.11,0.861138,https://realting.com/property-for-sale/greece/...
1,2,Finland,"Kittilae, Mainland, Tunturi-Lapin seutukunta, ...",34369.08,3285.24,0.858710,https://realting.com/property-for-sale/finland...
2,3,Italy,"Alessandria della Rocca, Sicily, Agrigento",32216.13,2280.71,0.783379,https://realting.com/property-for-sale/italy/a...
3,4,Russia,"Pudost, Northwestern Federal District, Pudosts...",30213.84,1561.17,0.740157,https://realting.com/property-for-sale/russia/...
4,5,Spain,"Valencian Community, Gandia, Safor",30407.28,1298.94,0.702864,https://realting.com/property-for-sale/spain/v...
5,6,Montenegro,"Sutomore, Bar Municipality",33358.17,1768.98,0.692765,https://realting.com/property-for-sale/montene...
6,7,Latvia,"Vidzeme, Riga",32347.26,1540.48,0.687791,https://realting.com/property-for-sale/latvia/...
7,8,Turkey,"Mediterranean Region, Sekerhane Mahallesi, Alanya",38144.88,1405.07,0.535587,https://realting.com/property-for-sale/turkey/...


Resultados: 8 | Paises unicos: 8


In [7]:
# DIAGNOSTICO: por que solo hay 8 resultados
diag = quick_df.copy()
total = len(diag)

diag['price_eur_num'] = pd.to_numeric(diag['price_in_Euro_calculo'], errors='coerce')
diag['rent_eur_month_num'] = parse_currency_quick(diag.get('zone_rental_price', pd.Series(index=diag.index, dtype='object')))

listing_text_d = (
    diag.get('location', pd.Series(index=diag.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + diag.get('url', pd.Series(index=diag.index, dtype='object')).fillna('').astype(str).str.lower()
)

sin_sold   = diag[~listing_text_d.str.contains(r'\bsold\b|\bvendido\b|\breservado\b', regex=True, na=False)]
con_pais   = sin_sold[sin_sold['country'].notna()]
con_precio = con_pais[con_pais['price_eur_num'].between(30000, 45000)]
con_renta  = con_precio[con_precio['rent_eur_month_num'] >= 1200]

print(f"Total registros           : {total:>8,}")
print(f"Tras quitar sold/vendido  : {len(sin_sold):>8,}")
print(f"Con pais valido           : {len(con_pais):>8,}")
print(f"Precio 30k-45k EUR        : {len(con_precio):>8,}")
print(f"Renta >= 1.200 EUR/mes    : {len(con_renta):>8,}  <- aqui se corta")
print()
# Cuanto tiene renta registrada (no NaN)
con_renta_any = con_precio[con_precio['rent_eur_month_num'].notna()]
print(f"De los {len(con_precio):,} con precio OK, tienen renta registrada: {len(con_renta_any):,}")
if len(con_renta_any):
    print(f"  Renta min: {con_renta_any['rent_eur_month_num'].min():.0f} EUR | max: {con_renta_any['rent_eur_month_num'].max():.0f} EUR | mediana: {con_renta_any['rent_eur_month_num'].median():.0f} EUR")
print()
# Cuantos paises hay si bajamos renta a 500
renta_500 = con_precio[con_precio['rent_eur_month_num'] >= 500]
print(f"Con renta >= 500 EUR/mes  : {len(renta_500):>8,}  ({renta_500['country'].nunique()} paises)")
renta_800 = con_precio[con_precio['rent_eur_month_num'] >= 800]
print(f"Con renta >= 800 EUR/mes  : {len(renta_800):>8,}  ({renta_800['country'].nunique()} paises)")
renta_1000 = con_precio[con_precio['rent_eur_month_num'] >= 1000]
print(f"Con renta >= 1.000 EUR/mes: {len(renta_1000):>8,}  ({renta_1000['country'].nunique()} paises)")
print(f"Con renta >= 1.200 EUR/mes: {len(con_renta):>8,}  ({con_renta['country'].nunique()} paises)")


Total registros           :  144,961
Tras quitar sold/vendido  :  144,961
Con pais valido           :  144,831
Precio 30k-45k EUR        :    5,234
Renta >= 1.200 EUR/mes    :      113  <- aqui se corta

De los 5,234 con precio OK, tienen renta registrada: 5,234
  Renta min: 110 EUR | max: 8132 EUR | mediana: 258 EUR

Con renta >= 500 EUR/mes  :    1,888  (14 paises)
Con renta >= 800 EUR/mes  :      459  (11 paises)
Con renta >= 1.000 EUR/mes:      155  (9 paises)
Con renta >= 1.200 EUR/mes:      113  (8 paises)


In [10]:
# TABLA DE RESULTADOS: pisos que cumplen filtro estricto (precio 30k-45k + renta >=1200)
from IPython.display import HTML
import pandas as pd

tabla = top20_rapido[[
    'rank_top20',
    'country',
    'location',
    'price_eur_num',
    'rent_eur_month_num',
    'score_objetivo',
    'url',
]].copy()

tabla = tabla.rename(columns={
    'rank_top20':         '#',
    'country':            'Pais',
    'location':           'Ubicacion',
    'price_eur_num':      'Precio (EUR)',
    'rent_eur_month_num': 'Renta/mes (EUR)',
    'score_objetivo':     'Score',
    'url':                'URL',
})

tabla['Precio (EUR)']    = tabla['Precio (EUR)'].map('{:,.0f}'.format)
tabla['Renta/mes (EUR)'] = tabla['Renta/mes (EUR)'].map('{:,.0f}'.format)
tabla['Score']           = (tabla['Score'] * 100).map('{:.1f}%'.format)
tabla['URL']             = tabla['URL'].apply(lambda u: f'<a href="{u}" target="_blank">ver</a>')

html = (
    tabla.style
        .set_caption('Filtro estricto: precio 30k–45k EUR | renta ≥ 1.200 EUR/mes')
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '6px')]},
            {'selector': 'th',      'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '6px 10px')]},
            {'selector': 'td',      'props': [('padding', '5px 10px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]},
        ])
        .hide(axis='index')
        .to_html(escape=False)
)

display(HTML(html))


#,Pais,Ubicacion,Precio (EUR),Renta/mes (EUR),Score,URL
1,Greece,"Macedonia and Thrace, Municipality of Paggaio, Eastern Macedonia and Thrace","30,204","2,527",86.1%,ver
2,Finland,"Kittilae, Mainland, Tunturi-Lapin seutukunta, Lapland","34,369","3,285",85.9%,ver
3,Italy,"Alessandria della Rocca, Sicily, Agrigento","32,216","2,281",78.3%,ver
4,Russia,"Pudost, Northwestern Federal District, Pudostskoe selskoe poselenie, Gatchinsky District","30,214","1,561",74.0%,ver
5,Spain,"Valencian Community, Gandia, Safor","30,407","1,299",70.3%,ver
6,Montenegro,"Sutomore, Bar Municipality","33,358","1,769",69.3%,ver
7,Latvia,"Vidzeme, Riga","32,347","1,540",68.8%,ver
8,Turkey,"Mediterranean Region, Sekerhane Mahallesi, Alanya","38,145","1,405",53.6%,ver


In [11]:
# TOP 20 AMPLIADO: mejor piso por pais para Streamlit
# Filtros: precio 30k-100k EUR | sin minimo de renta (renta entra en score)
PRECIO_MAX  = 100_000
PRECIO_OBJ  = 30_000   # precio ideal al que te quieres acercar
RENTA_OBJ   = 1_200    # renta ideal de referencia para puntuar

base = quick_df.copy()

# Quitar sold/reservados
lt = (
    base.get('location', pd.Series(index=base.index, dtype='object')).fillna('').astype(str).str.lower()
    + ' '
    + base.get('url', pd.Series(index=base.index, dtype='object')).fillna('').astype(str).str.lower()
)
sold_pat = r'\bsold\b|\bvendido\b|\bvendida\b|\breservado\b|\breserved\b'
base = base[~lt.str.contains(sold_pat, regex=True, na=False)].copy()

# Precio
base['price_eur_num']    = pd.to_numeric(base['price_in_Euro_calculo'], errors='coerce')
base['rent_eur_month_num'] = parse_currency_quick(base.get('zone_rental_price', pd.Series(index=base.index, dtype='object')))

candidatos = base[
    base['country'].notna()
    & base['price_eur_num'].between(PRECIO_OBJ, PRECIO_MAX, inclusive='both')
    & base['rent_eur_month_num'].notna()
].copy()

print(f'Candidatos con precio {PRECIO_OBJ:,}-{PRECIO_MAX:,} EUR y renta registrada: {len(candidatos):,} ({candidatos["country"].nunique()} paises)')

# Score 0-100
candidatos['score_precio'] = (1 - (candidatos['price_eur_num'] - PRECIO_OBJ).abs() / PRECIO_MAX).clip(0, 1)
candidatos['score_renta']  = (candidatos['rent_eur_month_num'] / RENTA_OBJ).clip(0, 1)
candidatos['score_total']  = (0.60 * candidatos['score_precio'] + 0.40 * candidatos['score_renta']) * 100

# Mejor piso por pais -> Top 20
top20_ampliado = (
    candidatos.sort_values('score_total', ascending=False)
              .groupby('country', as_index=False).head(1)
              .sort_values('score_total', ascending=False)
              .head(20)
              .reset_index(drop=True)
)
top20_ampliado.insert(0, 'rank_top20', np.arange(1, len(top20_ampliado) + 1))

# Tabla
tabla2 = top20_ampliado[[
    'rank_top20', 'country', 'location',
    'price_eur_num', 'rent_eur_month_num', 'score_total', 'url'
]].copy().rename(columns={
    'rank_top20':         '#',
    'country':            'Pais',
    'location':           'Ubicacion',
    'price_eur_num':      'Precio (EUR)',
    'rent_eur_month_num': 'Renta/mes (EUR)',
    'score_total':        'Score',
    'url':                'URL',
})
tabla2['Precio (EUR)']    = tabla2['Precio (EUR)'].map('{:,.0f}'.format)
tabla2['Renta/mes (EUR)'] = tabla2['Renta/mes (EUR)'].map('{:,.0f}'.format)
tabla2['Score']           = tabla2['Score'].map('{:.1f}%'.format)
tabla2['URL']             = tabla2['URL'].apply(lambda u: f'<a href="{u}" target="_blank">ver</a>')

html2 = (
    tabla2.style
        .set_caption(f'TOP 20 AMPLIADO: precio {PRECIO_OBJ:,}–{PRECIO_MAX:,} EUR | mejor piso por pais | score 0–100%')
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '6px')]},
            {'selector': 'th',      'props': [('background-color', '#1a5276'), ('color', 'white'), ('padding', '6px 10px')]},
            {'selector': 'td',      'props': [('padding', '5px 10px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#eaf4fb')]},
        ])
        .hide(axis='index')
        .to_html(escape=False)
)
display(HTML(html2))
print(f'\nResultados: {len(top20_ampliado)} paises | Score max: {top20_ampliado["score_total"].max():.1f}% | min: {top20_ampliado["score_total"].min():.1f}%')


Candidatos con precio 30,000-100,000 EUR y renta registrada: 31,605 (21 paises)


#,Pais,Ubicacion,Precio (EUR),Renta/mes (EUR),Score,URL
1,Greece,"Macedonia and Thrace, Municipality of Paggaio, Eastern Macedonia and Thrace","30,204","2,527",99.9%,ver
2,Russia,"Northwestern Federal District, Kaliningrad","30,214","1,561",99.9%,ver
3,Spain,"Valencian Community, Gandia, Safor","30,407","1,299",99.8%,ver
4,Italy,"Alessandria della Rocca, Sicily, Agrigento","32,216","2,281",98.7%,ver
5,Latvia,"Vidzeme, Riga","32,347","1,540",98.6%,ver
6,Montenegro,"Sutomore, Bar Municipality","33,358","1,769",98.0%,ver
7,Finland,"Kiukainen, Mainland, Rauman seutukunta, South-Western","30,325","1,137",97.7%,ver
8,Turkey,"Mediterranean Region, Sekerhane Mahallesi, Alanya","38,145","1,405",95.1%,ver
9,Czech Republic,"Velebudice, Northwest, okres Most, Ustecky kraj","31,499","1,008",92.7%,ver
10,Poland,"Masovian Voivodeship, Ostroleka","47,345","1,241",89.6%,ver



Resultados: 20 paises | Score max: 99.9% | min: 71.3%


In [12]:
import pathlib

out_path = pathlib.Path('../streamlit_data/top20_paises_verificables.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)

top20_ampliado.to_csv(out_path, index=False)
print(f'Exportado: {out_path.resolve()}')
print(f'Filas: {len(top20_ampliado)} | Paises: {list(top20_ampliado["country"])}')


Exportado: C:\Users\Usuario\OneDrive\Documentos\GitHub\Proyecto_Inmuebles\proyecto-inmobiliario-global\streamlit_data\top20_paises_verificables.csv
Filas: 20 | Paises: ['Greece', 'Russia', 'Spain', 'Italy', 'Latvia', 'Montenegro', 'Finland', 'Turkey', 'Czech Republic', 'Poland', 'Belarus', 'Uzbekistan', 'Lithuania', 'Northern Cyprus', 'UAE', 'Thailand', 'Hungary', 'Croatia', 'Cyprus', 'Georgia']
